In [ ]:
# ── Colab Setup (run this first) ──────────────────────────────────────────
!pip install openai PyMuPDF tiktoken -q

import os
# Option 1: Paste your key directly
os.environ["OPENAI_API_KEY"] = "sk-..."

# Option 2: Use Colab Secrets (recommended)
# from google.colab import userdata
# os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

Notebook 01 — Data Preparation
AutoRFP-AI: Generate Synthetic RFP Dataset

WHAT THIS DOES:
- Generates 50 realistic RFP question-answer pairs (the knowledge base)
- Creates a sample 12-question RFP document as PDF (the demo input)

COLAB SETUP — paste this in your first cell:
import os
os.environ["OPENAI_API_KEY"] = "sk-..."

### Setup

In [ ]:
import os
import json
import hashlib
from pathlib import Path
from openai import OpenAI

client = OpenAI()
DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)
print("Setup complete")

### Define RFP categories

In [ ]:
CATEGORIES = [
    {"name": "Security & Compliance",
     "description": "SOC 2, ISO 27001, encryption, access controls, vulnerability management",
     "count": 10},
    {"name": "Technical Architecture",
     "description": "System design, scalability, uptime, integrations, APIs, data flow",
     "count": 10},
    {"name": "Data Privacy & Governance",
     "description": "GDPR, CCPA, data residency, retention policies, DPA terms",
     "count": 8},
    {"name": "Implementation & Onboarding",
     "description": "Deployment timeline, migration, training, support during rollout",
     "count": 7},
    {"name": "Pricing & Commercial Terms",
     "description": "Licensing models, volume discounts, contract flexibility, payment terms",
     "count": 7},
    {"name": "Support & SLAs",
     "description": "Response times, escalation paths, dedicated account management, uptime guarantees",
     "count": 8},
]

TOTAL = sum(c["count"] for c in CATEGORIES)
print(f"Will generate {TOTAL} Q&A pairs across {len(CATEGORIES)} categories")

### Generate Q&A pairs

In [ ]:
SYSTEM_PROMPT = """You are an expert enterprise sales engineer at Acme Cloud Platform,
a B2B SaaS company providing cloud-based data analytics and workflow automation.

Generate realistic RFP questions that enterprise buyers actually ask, paired with
polished winning answers.

QUESTION RULES:
- Specific and detailed, like real procurement teams write
- Include sub-parts where appropriate
- Vary the complexity

ANSWER RULES:
- Professional but not robotic
- Include specific (fictional but realistic) details: certifications, SLA numbers, feature names
- 80-200 words each
- Reference "Acme Cloud Platform" occasionally
- Include concrete numbers (99.95% uptime, AES-256, etc.)

Return ONLY valid JSON. No markdown fences."""


def generate_qa_pairs(category_name, category_desc, count):
    """Generate Q&A pairs for one RFP category."""
    user_prompt = f"""Generate exactly {count} RFP question-and-answer pairs for:
"{category_name}" — topics: {category_desc}

Return a JSON array:
[{{"question": "...", "answer": "...", "category": "{category_name}"}}]

ONLY the JSON array. No other text."""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt},
        ],
        temperature=0.8, max_tokens=4000,
    )
    raw = response.choices[0].message.content.strip()
    if raw.startswith("```"):
        raw = raw.split("\n", 1)[1].rsplit("```", 1)[0]
    return json.loads(raw)


all_qa_pairs = []
for cat in CATEGORIES:
    print(f"  Generating {cat['count']} pairs for: {cat['name']}...")
    pairs = generate_qa_pairs(cat["name"], cat["description"], cat["count"])
    all_qa_pairs.extend(pairs)
    print(f"    Got {len(pairs)} pairs")

# Add unique IDs and content hashes
for i, pair in enumerate(all_qa_pairs):
    pair["id"] = f"QA-{i+1:03d}"
    pair["hash"] = hashlib.md5(pair["answer"].encode()).hexdigest()[:8]

print(f"\nTotal Q&A pairs: {len(all_qa_pairs)}")

### Save the knowledge base

In [ ]:
kb_path = DATA_DIR / "past_responses.json"
with open(kb_path, "w") as f:
    json.dump(all_qa_pairs, f, indent=2)

print(f"Saved knowledge base to {kb_path}")

# Preview
sample = all_qa_pairs[0]
print(f"\n-- Sample --")
print(f"Category: {sample['category']}")
print(f"Q: {sample['question'][:120]}...")
print(f"A: {sample['answer'][:120]}...")

### Generate sample RFP document (PDF)

In [ ]:
SAMPLE_RFP_PROMPT = """Generate a realistic RFP document from "GlobalTech Enterprises"
seeking a cloud analytics platform vendor.

Include:
1. Brief intro (2-3 sentences about GlobalTech and what they need)
2. Exactly 12 numbered questions:
   - Security (3 questions)
   - Technical Architecture (3 questions)
   - Data Privacy (2 questions)
   - Implementation (2 questions)
   - Pricing (2 questions)

Make questions detailed and realistic. Some should have sub-parts.
Format as plain text with clear section headers. Number each question.
Start with: "REQUEST FOR PROPOSAL — Cloud Analytics Platform"
Include: "Issued by: GlobalTech Enterprises"
and "Response Deadline: [14 business days from receipt]"

Return ONLY the document text."""

print("Generating sample RFP...")
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": SAMPLE_RFP_PROMPT}],
    temperature=0.7, max_tokens=3000,
)
rfp_text = response.choices[0].message.content.strip()

# Save as PDF using PyMuPDF
import fitz

def text_to_pdf(text, output_path):
    """Convert plain text to a clean PDF."""
    doc = fitz.open()
    lines = text.split("\n")
    page_w, page_h = 612, 792
    margin = 72
    y = margin
    page = doc.new_page(width=page_w, height=page_h)

    for line in lines:
        if y > page_h - margin - 20:
            page = doc.new_page(width=page_w, height=page_h)
            y = margin

        if line.strip() == "":
            y += 7
            continue

        is_header = line.startswith("REQUEST FOR") or line.startswith("Section") or line.isupper()
        fs = 14 if is_header else 11

        rect = fitz.Rect(margin, y, page_w - margin, y + 200)
        page.insert_textbox(rect, line, fontsize=fs, fontname="helv")

        chars_per_line = int((page_w - 2 * margin) / (fs * 0.5))
        num_lines = max(1, len(line) // chars_per_line + 1)
        y += 14 * num_lines + 4

    doc.save(output_path)
    doc.close()

pdf_path = str(DATA_DIR / "sample_rfp.pdf")
text_to_pdf(rfp_text, pdf_path)
print(f"Saved sample RFP to {pdf_path}")

# Also save raw text for easy loading in notebook 03
with open(DATA_DIR / "sample_rfp_text.txt", "w") as f:
    f.write(rfp_text)

print(f"\n-- RFP Preview (first 400 chars) --")
print(rfp_text[:400])

### Verify

In [ ]:
print("\n" + "=" * 50)
print("DATA PREPARATION COMPLETE")
print("=" * 50)
for f in sorted(DATA_DIR.iterdir()):
    print(f"  {f.name} ({f.stat().st_size:,} bytes)")
print(f"\nKnowledge base: {len(all_qa_pairs)} Q&A pairs")
print("Ready for Notebook 02 (Vector Store)")